# 00 · Environment, persistent paths and recorded permissions

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Start here after installation. No GPU or new data are required to initialise the workspace.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Install compatible research dependencies
Keep Colab’s matched PyTorch/TorchVision/CUDA stack; do not force a CUDA wheel replacement.

In [ ]:
import subprocess
INSTALL_DEPENDENCIES = ON_COLAB  # False in local CI; switch on for a fresh Colab runtime.
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)+"[test,report,app]"], check=True)
print(subprocess.run([sys.executable,"-m","pip","check"], capture_output=True, text=True).stdout)

## 2. Inspect actual runtime resources
Your 5 TB Drive allocation is persistent storage, not GPU memory or Colab local disk. No assumed A100/H100 or unlimited runtime.

In [ ]:
import torch, torchvision, shutil
p=initialize(cfg)
print({"python":sys.version,"torch":torch.__version__,"torchvision":torchvision.__version__,"cuda":torch.cuda.is_available(),"local_disk_GiB":round(shutil.disk_usage('/content' if ON_COLAB else '/tmp').free/2**30,1)})
if torch.cuda.is_available():
    props=torch.cuda.get_device_properties(0)
    print({"GPU":props.name,"VRAM_GiB":round(props.total_memory/2**30,1)})

## 3. Show actual approval requirements
Do not mark pending decisions approved merely to make the code run. The synthetic demo needs no research approval.

In [ ]:
from oncoplate.governance import approval_report
approvals=approval_report(cfg['root'])
print(json.dumps(approvals,indent=2))
print("Actual decision records belong at:",p['permissions']/"approvals.json")

## 4. Check software integrity before running expensive jobs

In [ ]:
subprocess.run([sys.executable,"-m","pytest","-q",str(REPO/'tests')],cwd=REPO,check=True)
subprocess.run([sys.executable,str(REPO/'scripts/verify_notebooks.py')],cwd=REPO,check=True)

## 5. Inspect active-study configuration
FoodNExTDB is the initial foundation. Notebook 07 explicitly switches to the new documented benchmark.

In [ ]:
write_json(p['reports']/"resolved_config.json",cfg)
print(json.dumps({k:str(v) for k,v in p.items()},indent=2))
print("Next: notebook 90 for the isolated software demo, then 01 for authorised public data.")

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
